# Synthetic signal validation

## Research question

Can SBX PPO learn a known state-dependent exposure when a signal exists, while avoiding a systematic false advantage when the signal is zero?

In [ ]:
from pathlib import Path
from collections import deque
import json
import re
import subprocess
import sys
import pandas as pd
from IPython.display import Image, display

START = Path.cwd().resolve()
ROOT = next(
    (candidate for candidate in (START, *START.parents) if (candidate / "pyproject.toml").is_file()),
    None,
)
assert ROOT is not None, "Could not locate the project root containing pyproject.toml."
print(f"Python: {sys.version.split()[0]}")
print(f"Project: {ROOT}")

## Inputs and provenance

The two conditions share the generator and cost rules; only signal strength changes. Fixed local validation and test paths prevent result drift.

In [ ]:
CONFIGS = ['configs/experiments/synthetic_high.yaml', 'configs/experiments/synthetic_zero.yaml']
SEEDS = [101, 202, 303]
RUN_EXPERIMENTS = False
print({"run_experiments": RUN_EXPERIMENTS, "models_in_this_notebook": len(CONFIGS) * len(SEEDS),
       "steps_per_model": 5_001_216, "configs": CONFIGS, "seeds": SEEDS})

## Data checks

In [ ]:
from dtasrl.config import load_config
from dtasrl.storage import verify_inventory
rows = []
for path in CONFIGS:
    cfg = load_config(ROOT / path)
    frozen = ROOT / cfg["data"]["frozen_inputs"]
    verify_inventory(frozen)
    rows.append({"config": path, "backend": cfg["backend"], "steps": cfg["total_timesteps"],
                 "frozen input": str(frozen.relative_to(ROOT)), "inventory verified": True})
display(pd.DataFrame(rows))

## Configuration

In [ ]:
display(pd.DataFrame([{"condition": Path(path).stem, **{key: load_config(ROOT / path).get(key) for key in ("kind", "algorithm", "total_timesteps", "transaction_cost_bps")}} for path in CONFIGS]))

## Execute experiments

The default is report-only. Set `RUN_EXPERIMENTS = True` to start the real 5,001,216-transition SBX/JAX training manually. Completed runs are skipped and matching interrupted runs resume from their latest safe checkpoint.

In [ ]:
from dtasrl.provenance import run_directory

execution_log = []
if RUN_EXPERIMENTS:
    # Refresh the immutable task plan against the current source before training.
    subprocess.run(
        [sys.executable, "-m", "dtasrl.cli", "study", "plan"],
        check=True,
        cwd=ROOT,
    )
    for config_path in CONFIGS:
        config = load_config(ROOT / config_path)
        for seed in SEEDS:
            run_path = run_directory(config, seed)
            complete = (run_path / "manifest.json").is_file() and (run_path / "metrics.json").is_file()
            if complete:
                execution_log.append({"config": config_path, "seed": seed, "action": "skipped_complete", "run": str(run_path)})
                print(f"SKIP complete | {Path(config_path).stem} | seed {seed}")
                continue
            command = [
                sys.executable, "-u", "-m", "dtasrl.cli", "experiment", "run",
                "--config", config_path, "--seed", str(seed),
            ]
            if run_path.exists():
                resumable = (
                    (run_path / "training_identity.json").is_file()
                    and (run_path / "latest_checkpoint.json").is_file()
                )
                if not resumable:
                    raise RuntimeError(
                        f"Incomplete but non-resumable run found at {run_path}. "
                        "Inspect it before removing or archiving it."
                    )
                command.append("--resume")
                action = "resumed"
            else:
                action = "started"
            print(f"{action.upper()} | {Path(config_path).stem} | seed {seed}", flush=True)
            process = subprocess.Popen(
                command,
                cwd=ROOT,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                encoding="utf-8",
                errors="replace",
                bufsize=1,
            )
            assert process.stdout is not None
            # SB3's Rich/tqdm bar writes carriage-return frames. Replace one
            # notebook display instead of printing every refresh as a new line.
            progress_display = display("Starting SB3 progress bar...", display_id=True)
            recent_output = deque(maxlen=80)
            ansi = re.compile(r"\x1B(?:[@-Z\\-_]|\[[0-?]*[ -/]*[@-~])")
            frame = []
            while character := process.stdout.read(1):
                if character in ("\r", "\n"):
                    rendered = ansi.sub("", "".join(frame)).strip()
                    frame.clear()
                    if rendered:
                        recent_output.append(rendered)
                        if progress_display is not None:
                            progress_display.update(rendered)
                else:
                    frame.append(character)
            rendered = ansi.sub("", "".join(frame)).strip()
            if rendered:
                recent_output.append(rendered)
            return_code = process.wait()
            if return_code:
                print("\n".join(recent_output))
                raise subprocess.CalledProcessError(return_code, command)
            if progress_display is not None:
                progress_display.update(f"COMPLETED | {Path(config_path).stem} | seed {seed}")
            execution_log.append({"config": config_path, "seed": seed, "action": action, "run": str(run_path)})
display(pd.DataFrame(execution_log))

## Results

In [ ]:
from dtasrl.study_reports import control_example, experiment_report
report = experiment_report("01_synthetic")
worked, worked_figure = control_example("01_synthetic")
display(report["status"])
if report["summary"].empty:
    print("Formal results: not_started. No PPO metric is shown.")
else:
    display(report["summary"])
display(worked)

## Figures

In [ ]:
for path in report["figures"]:
    display(Image(filename=str(path)))
if report["summary"].empty:
    display(Image(filename=str(worked_figure)))
    print("Worked example only; it does not use a trained model.")
else:
    print(f"Detailed run figures saved locally: {len(report['detail_figures'])}")

## Interpretation and limitations

The hand-chosen worked example validates accounting and counterfactual semantics only. It is deliberately separated from formal runs. Three learner seeds describe optimization variability, not three independent markets.

## Output inventory

In [ ]:
display(pd.read_csv(ROOT / 'outputs/01_synthetic/inventory.csv'))